In [15]:
# ─── Cell 1: Installs ────────────────────────────────────────
%pip install alpaca-py
%pip install pandas numpy python-dotenv

# ─── Cell 1: Installs + Morning Workflow ─────────────────────
# ============================================================
# 200-EMA MOMO COUNTER-TREND OPENING DRIVE
# Alert Tracker + Paper Trading
# ============================================================
#
# EVERY MORNING — DO THIS BEFORE RUNNING:
# ─────────────────────────────────────────
# 1. Open 200_watchlist.txt
# 2. Add today's tickers from TraderTV/Walter Bloomberg
#    (one per line, no spaces):
#    SNAP
#    MRVL
#    DKNG
#    PLTU
# 3. Save 200_watchlist.txt
# 4. Come back here
# 5. Run All Cells (Ctrl+F9 or click Run All)
# 6. Cell 13 starts the tracker automatically
#
# WHAT HAPPENS AUTOMATICALLY:
# ─────────────────────────────────────────
# 5:00AM - 9:30AM → Pre-market EMA touches logged to CSV
# 9:30AM - 9:40AM → Opening drive alerts fire
#                   Score 4-5 → Paper trade placed automatically
#                   Score 1-3 → Alert logged only
# After 9:40AM    → Alert window closes
#                   Open positions monitored until exit
# 4:00PM          → All positions closed — no overnight holds
#
# AFTER MARKET:
# ─────────────────────────────────────────
# Run Cell 12 → view_alerts() to review today's setups
# Open 200_EMA_MOMO_alerts.csv in Excel
# Fill in: outcome / entry_price / exit_price / pnl / notes
# Your research database grows every trading day
#
# SCORING REMINDER:
# ─────────────────────────────────────────
# +1 → 5-MIN 200-EMA touched
# +1 → 2-MIN 200-EMA touched
# +1 → Both EMAs converging at same level
# +1 → VWAP near 2-MIN EMA
# +1 → Pre-market low near either EMA
# ─────────────────────────────────────────
# Score 4-5 → Paper trade placed
# Score 1-3 → Alert only (study in ToS playback)
# Max 2 attempts per stock per day
# ============================================================




Note: you may need to restart the kernel to use updated packages.


In [16]:
# ─── Cell 2: Imports ─────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import time as time_module
import datetime as dt
from datetime import datetime
from dotenv import load_dotenv

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [17]:
# ─── Cell 3: Alpaca Connection ────────────────────────────────
dotenv_path = r"c:\Users\kwgit\Desktop\Playbook\Claude_2025_2026_Playbooks\AlgoTrading\AlgoTrading\.env"
load_dotenv(dotenv_path, override=True)

API_KEY          = os.getenv("ALPACA_API_KEY")
SECRET_KEY       = os.getenv("ALPACA_SECRET_KEY")
PAPER_API_KEY    = os.getenv("ALPACA_PAPER_API_KEY")
PAPER_SECRET_KEY = os.getenv("ALPACA_PAPER_SECRET_KEY")

# Data client — live keys for market data
client = StockHistoricalDataClient(
    api_key    = API_KEY,
    secret_key = SECRET_KEY
)

# Trading client — paper keys for execution
trading_client = TradingClient(
    api_key    = PAPER_API_KEY,
    secret_key = PAPER_SECRET_KEY,
    paper      = True
)

account = trading_client.get_account()

print("✅ Alpaca connected")
print(f"   Data client:     ✅ Live keys")
print(f"   Trading client:  ✅ Paper keys")
print(f"   Cash:            ${float(account.cash):,.2f}")
print(f"   Paper Trading:   ✅ Confirmed")

✅ Alpaca connected
   Data client:     ✅ Live keys
   Trading client:  ✅ Paper keys
   Cash:            $10,130.20
   Paper Trading:   ✅ Confirmed


In [18]:
# ─── Cell 4: Configuration ────────────────────────────────────

# ── Alert distance threshold ──────────────────────────────────
EMA_ALERT_PCT        = 0.005   # 0.5% of price
EMA_ALERT_MIN_DOLLAR = 0.15    # Minimum $0.15 regardless of %

# ── VWAP confluence threshold ─────────────────────────────────
VWAP_CONFLUENCE_PCT  = 0.005   # 0.5%

# ── EMA convergence threshold ─────────────────────────────────
EMA_CONVERGENCE_PCT  = 0.005   # 0.5%

# ── Pre-market level confluence ───────────────────────────────
PM_CONFLUENCE_PCT    = 0.005   # 0.5%

# ── Session times ─────────────────────────────────────────────
PREMARKET_START      = dt.time(5, 0)
MARKET_OPEN          = dt.time(9, 30)
ALERT_WINDOW_END     = dt.time(9, 40)
EOD                  = dt.time(16, 0)

# ── Dollar risk per trade ─────────────────────────────────────
DOLLAR_RISK          = 100     # $100 = 1% of $10K account

# ── File paths ────────────────────────────────────────────────
CSV_FILE       = r"c:\Users\kwgit\Desktop\Playbook\Claude_2025_2026_Playbooks\AlgoTrading\AlgoTrading\200_EMA_MOMO_alerts.csv"
WATCHLIST_FILE = r"c:\Users\kwgit\Desktop\Playbook\Claude_2025_2026_Playbooks\AlgoTrading\AlgoTrading\200_watchlist.txt"

print("✅ Configuration loaded")
print(f"   EMA alert threshold: {EMA_ALERT_PCT*100}% "
      f"(min ${EMA_ALERT_MIN_DOLLAR})")
print(f"   VWAP confluence:     {VWAP_CONFLUENCE_PCT*100}%")
print(f"   EMA convergence:     {EMA_CONVERGENCE_PCT*100}%")
print(f"   Pre-market start:    {PREMARKET_START}")
print(f"   Alert window:        {MARKET_OPEN} - {ALERT_WINDOW_END}")
print(f"   Dollar risk:         ${DOLLAR_RISK}/trade")
print(f"   CSV:                 {CSV_FILE}")
print(f"   Watchlist:           {WATCHLIST_FILE}")

✅ Configuration loaded
   EMA alert threshold: 0.5% (min $0.15)
   VWAP confluence:     0.5%
   EMA convergence:     0.5%
   Pre-market start:    05:00:00
   Alert window:        09:30:00 - 09:40:00
   Dollar risk:         $100/trade
   CSV:                 c:\Users\kwgit\Desktop\Playbook\Claude_2025_2026_Playbooks\AlgoTrading\AlgoTrading\200_EMA_MOMO_alerts.csv
   Watchlist:           c:\Users\kwgit\Desktop\Playbook\Claude_2025_2026_Playbooks\AlgoTrading\AlgoTrading\200_watchlist.txt


In [19]:
# ─── Cell 5: Setup Files ──────────────────────────────────────

# ── Create 200_watchlist.txt if it doesn't exist ─────────────
if not os.path.exists(WATCHLIST_FILE):
    with open(WATCHLIST_FILE, 'w') as f:
        f.write("# Add today's tickers one per line\n")
        f.write("# Lines starting with # are ignored\n")
        f.write("# Example:\n")
        f.write("# SNAP\n")
        f.write("# MRVL\n")
        f.write("# DKNG\n")
    print(f"✅ Created 200_watchlist.txt")
    print(f"   Open and add today's tickers before running")
else:
    with open(WATCHLIST_FILE, 'r') as f:
        lines = [l.strip() for l in f
                 if l.strip() and not l.startswith('#')]
    print(f"✅ 200_watchlist.txt found")
    print(f"   Current tickers: {lines}")

# ── Create CSV with headers if it doesn't exist ──────────────
csv_headers = [
    # Timing
    'timestamp', 'ticker', 'alert_type', 'attempt_count',

    # Price
    'current_price', 'open_price',

    # 5-MIN 200-EMA
    'ema200_5min', 'dist_to_5min_ema_pct',
    '5min_ema_alert',
    'vwap_confluence_5min', 'pm_low_confluence_5min',
    'pm_high_confluence_5min',

    # 2-MIN 200-EMA
    'ema200_2min', 'dist_to_2min_ema_pct',
    '2min_ema_alert',
    'vwap_confluence_2min', 'pm_low_confluence_2min',
    'pm_high_confluence_2min',

    # VWAP
    'vwap', 'dist_5min_to_vwap_pct', 'dist_2min_to_vwap_pct',

    # Pre-market
    'premarket_high', 'premarket_low',
    'pm_touch_5min', 'pm_touch_2min',
    'pm_touch_time', 'pm_touch_price',

    # EMA convergence
    'emas_converging', 'ema_convergence_dist_pct',

    # 9-EMA triple confluence
    'ema9_2min', 'ema9_confluence',

    # Setup type ← NEW
    'setup_type',    # 'standard' or 'blue_sky'

    # Conviction
    'conviction_score',

    # You fill in after market
    'paper_traded',
    'outcome', 'entry_price', 'exit_price', 'pnl', 'notes'
]

if not os.path.exists(CSV_FILE):
    pd.DataFrame(columns=csv_headers).to_csv(
        CSV_FILE, index=False)
    print(f"✅ Created 200_EMA_MOMO_alerts.csv")
else:
    existing = pd.read_csv(CSV_FILE)
    print(f"✅ CSV found — {len(existing)} alerts logged so far")

    # Add setup_type column if missing
    if 'setup_type' not in existing.columns:
        existing['setup_type'] = 'standard'
        existing.to_csv(CSV_FILE, index=False)
        print(f"   ✅ Added setup_type column to existing CSV")

✅ 200_watchlist.txt found
   Current tickers: ['SNAP', 'MRVL', 'DKNG', 'PLTU', 'SMCX', 'NVDL', 'SNXX', 'AMZU', 'MSFU', 'AAPU', 'TSLL', 'SPCH', 'SSPC', 'MRNA', 'SKHX', 'MVLL', 'IBIT', 'CONL', 'AMDL', 'HOOG', 'SNDQ', 'CMG', 'CRMG', 'AFRM', 'GME']
✅ CSV found — 567 alerts logged so far


In [20]:
# ─── Cell 6: Watchlist Loader ─────────────────────────────────

def load_watchlist():
    """
    Read today's tickers from 200_watchlist.txt
    Lines starting with # are comments — ignored
    Empty lines ignored
    Updates automatically every scan
    """
    if not os.path.exists(WATCHLIST_FILE):
        print("❌ 200_watchlist.txt not found")
        return []

    tickers = []
    with open(WATCHLIST_FILE, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                tickers.append(line.upper())
    return tickers


# Test
watchlist = load_watchlist()
print(f"✅ Watchlist loader ready")
print(f"   Current watchlist: {watchlist}")
print(f"\n   To update: edit 200_watchlist.txt and save")

✅ Watchlist loader ready
   Current watchlist: ['SNAP', 'MRVL', 'DKNG', 'PLTU', 'SMCX', 'NVDL', 'SNXX', 'AMZU', 'MSFU', 'AAPU', 'TSLL', 'SPCH', 'SSPC', 'MRNA', 'SKHX', 'MVLL', 'IBIT', 'CONL', 'AMDL', 'HOOG', 'SNDQ', 'CMG', 'CRMG', 'AFRM', 'GME']

   To update: edit 200_watchlist.txt and save


In [21]:
# ─── Cell 7: Data Fetcher ─────────────────────────────────────

def fetch_bars(ticker, timeframe_minutes, lookback_days=5):
    """
    Fetch bars for one ticker on one timeframe
    Pulls 5 days back to ensure 200 bars for reliable EMA
    200-EMA needs at least 200 bars to be accurate
    """
    try:
        start = datetime.now() - dt.timedelta(
            days=lookback_days)

        req = StockBarsRequest(
            symbol_or_symbols = [ticker],
            timeframe          = TimeFrame(
                timeframe_minutes,
                TimeFrameUnit.Minute
            ),
            start = start,
        )

        bars = client.get_stock_bars(req)
        df   = bars.df.reset_index()

        if len(df) == 0:
            return None

        # Clean timestamps
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        if df['timestamp'].dt.tz is None:
            df['timestamp'] = df['timestamp'] \
                .dt.tz_localize('UTC')
        df['timestamp'] = df['timestamp'] \
            .dt.tz_convert('America/New_York')
        df = df.set_index('timestamp')
        df = df.sort_index()

        return df

    except Exception as e:
        print(f"   ❌ {ticker} {timeframe_minutes}MIN: {e}")
        return None


def get_premarket_data(df):
    """
    Extract pre-market data (5AM - 9:30AM) for today

    Fix: Build timezone-aware timestamp directly
    instead of localizing after replace()
    which strips timezone in some pandas versions
    """
    today    = datetime.now().date()
    pm_start = pd.Timestamp(
        year   = today.year,
        month  = today.month,
        day    = today.day,
        hour   = 5,
        minute = 0,
        tz     = 'America/New_York'
    )
    pm_end = pd.Timestamp(
        year   = today.year,
        month  = today.month,
        day    = today.day,
        hour   = 9,
        minute = 30,
        tz     = 'America/New_York'
    )
    return df[(df.index >= pm_start) & (df.index < pm_end)]


def get_market_data(df):
    """
    Extract regular market hours data (9:30AM+) for today

    Fix: Build timezone-aware timestamp directly
    instead of localizing after replace()
    which strips timezone in some pandas versions

    This was causing 0 bars returned at 9:30AM
    even when bars existed — the timezone comparison
    was failing silently
    """
    today     = datetime.now().date()
    mkt_start = pd.Timestamp(
        year   = today.year,
        month  = today.month,
        day    = today.day,
        hour   = 9,
        minute = 30,
        tz     = 'America/New_York'
    )
    return df[df.index >= mkt_start]


print("✅ Data fetcher loaded")
print("   fetch_bars(ticker, timeframe_minutes)")
print("   get_premarket_data(df) — timezone fix applied")
print("   get_market_data(df)    — timezone fix applied")
print()
print("   KEY FIX (Aug 20 2026):")
print("   pd.Timestamp built with tz= parameter directly")
print("   Avoids timezone strip bug from")
print("   .tz_localize().replace() pattern")

✅ Data fetcher loaded
   fetch_bars(ticker, timeframe_minutes)
   get_premarket_data(df) — timezone fix applied
   get_market_data(df)    — timezone fix applied

   KEY FIX (Aug 20 2026):
   pd.Timestamp built with tz= parameter directly
   Avoids timezone strip bug from
   .tz_localize().replace() pattern


In [22]:
# ─── Cell 8: Alert Logic ──────────────────────────────────────

def calculate_200_ema(df):
    """
    Calculate 200-period EMA
    adjust=False matches ToS calculation exactly
    Returns None if fewer than 200 bars available
    """
    if len(df) < 200:
        return None
    df = df.copy()
    df['ema200'] = df['close'].ewm(span=200, adjust=False).mean()
    return df['ema200'].iloc[-1]


def calculate_9_ema(df):
    """
    Calculate 9-period EMA
    Used to detect triple confluence:
    9-EMA + 200-EMA + VWAP all at same level
    adjust=False matches ToS exactly
    """
    if len(df) < 9:
        return None
    df = df.copy()
    df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()
    return df['ema9'].iloc[-1]


def is_near_level(price, level, pct_threshold, min_dollar):
    """
    Check if price is within threshold of a level
    Uses whichever is larger: % or minimum dollar amount
    Protects low-priced stocks from too-tight thresholds
    """
    if level is None or level == 0:
        return False
    threshold = max(price * pct_threshold, min_dollar)
    distance  = abs(price - level)
    return distance <= threshold


def pct_distance(price, level):
    """Calculate % distance between two price levels"""
    if level is None or level == 0:
        return None
    return round(abs(price - level) / level * 100, 3)


def calculate_conviction_score(alert_data):
    """
    Score each setup 0-6
    Higher score = more confluence = higher conviction

    +1 → 5-MIN 200-EMA touched
    +1 → 2-MIN 200-EMA touched
    +1 → Both 200-EMAs converging at same level
    +1 → VWAP near 2-MIN EMA (2-MIN only)
    +1 → Pre-market low near either EMA
    +1 → 9-EMA near 200-EMA (triple confluence) ← NEW

    Note: pre-market touch tracked but NOT scored
    Add back later if data supports it

    New max score: 6
    Trade threshold: 5-6
    """
    score = 0
    if alert_data.get('5min_ema_alert'):              score += 1
    if alert_data.get('2min_ema_alert'):              score += 1
    if alert_data.get('emas_converging'):             score += 1
    if alert_data.get('vwap_confluence_2min'):        score += 1
    if (alert_data.get('pm_low_confluence_5min') or
        alert_data.get('pm_low_confluence_2min')):    score += 1
    if alert_data.get('ema9_confluence'):             score += 1
    return score


def determine_alert_type(alert_data):
    """
    Classify the setup:
    Type 1 → 5-MIN EMA only
    Type 2 → 2-MIN EMA only
    Type 3 → Both 200-EMAs converging
    Type 4 → EMA + VWAP confluence (2-MIN)
    Type 5 → EMA + pre-market low confluence
    Type 6 → Triple confluence (9-EMA + 200-EMA + VWAP) ← NEW
    """
    has_5min   = alert_data.get('5min_ema_alert', False)
    has_2min   = alert_data.get('2min_ema_alert', False)
    has_conv   = alert_data.get('emas_converging', False)
    has_vwap   = alert_data.get('vwap_confluence_2min', False)
    has_pm     = (alert_data.get('pm_low_confluence_5min', False) or
                  alert_data.get('pm_low_confluence_2min', False))
    has_triple = alert_data.get('ema9_confluence', False)

    if has_triple:                              return 6
    elif has_pm and (has_5min or has_2min):    return 5
    elif has_vwap and (has_5min or has_2min):  return 4
    elif has_conv:                             return 3
    elif has_5min and has_2min:                return 3
    elif has_5min:                             return 1
    elif has_2min:                             return 2
    else:                                      return 0


print("✅ Alert logic loaded")
print("   Scoring (max 6):")
print("   +1 → 5-MIN 200-EMA touched")
print("   +1 → 2-MIN 200-EMA touched")
print("   +1 → Both 200-EMAs converging")
print("   +1 → VWAP near 2-MIN EMA")
print("   +1 → Pre-market low near EMA")
print("   +1 → 9-EMA near 200-EMA (triple confluence)")
print()
print("   Alert types:")
print("   Type 6 → Triple confluence (A+ setup)")
print("   Type 5 → EMA + pre-market low")
print("   Type 4 → EMA + VWAP")
print("   Type 3 → Both EMAs converging")
print("   Type 2 → 2-MIN only")
print("   Type 1 → 5-MIN only")

✅ Alert logic loaded
   Scoring (max 6):
   +1 → 5-MIN 200-EMA touched
   +1 → 2-MIN 200-EMA touched
   +1 → Both 200-EMAs converging
   +1 → VWAP near 2-MIN EMA
   +1 → Pre-market low near EMA
   +1 → 9-EMA near 200-EMA (triple confluence)

   Alert types:
   Type 6 → Triple confluence (A+ setup)
   Type 5 → EMA + pre-market low
   Type 4 → EMA + VWAP
   Type 3 → Both EMAs converging
   Type 2 → 2-MIN only
   Type 1 → 5-MIN only


In [23]:
# ─── Cell 9: Alert Generator ──────────────────────────────────

daily_attempts = {}

def reset_daily_attempts():
    global daily_attempts
    daily_attempts = {}
    print("✅ Daily attempt counters reset")


def check_ticker(ticker):
    """
    Full 200-EMA analysis for one ticker
    Checks both 5-MIN and 2-MIN timeframes
    Also checks 9-EMA for triple confluence

    Attempt counter rules:
      Pre-market (5AM-9:30AM):
        → Alert and log only
        → Never counts as attempt
        → Never places order
        → Unlimited pre-market scans

      Market hours (9:30-9:40AM):
        → Paper trade window
        → Max 2 attempts enforced here
        → Orders placed if score 4+

    Blue Sky detection:
        → NOT done here
        → Done in place_momo_trade() at 9:30AM
        → When we have actual open price
        → Set to 'standard' here as default
    """
    global daily_attempts

    # ── Check attempt limit — market hours only ───────────────
    current_time = datetime.now().time()
    attempts     = daily_attempts.get(ticker, 0)


    # ── Fetch both timeframes ─────────────────────────────────
    df_5min = fetch_bars(ticker, 5)
    df_2min = fetch_bars(ticker, 2)

    if df_5min is None or df_2min is None:
        return None
    if len(df_5min) < 200 or len(df_2min) < 200:
        return None

    # ── Calculate 200-EMA on both timeframes ──────────────────
    ema_5min = calculate_200_ema(df_5min)
    ema_2min = calculate_200_ema(df_2min)

    if ema_5min is None or ema_2min is None:
        return None

    # ── Calculate 9-EMA on 2-MIN ──────────────────────────────
    ema9_2min = calculate_9_ema(df_2min)

    # ── Current price and VWAP ────────────────────────────────
    current_price = df_5min['close'].iloc[-1]
    vwap          = df_5min['vwap'].iloc[-1] \
                    if 'vwap' in df_5min.columns else None

    # ── Open price ────────────────────────────────────────────
    mkt_data   = get_market_data(df_5min)
    open_price = mkt_data['open'].iloc[0] \
                 if len(mkt_data) > 0 else current_price

    # ── Pre-market data ───────────────────────────────────────
    pm_5min = get_premarket_data(df_5min)
    pm_2min = get_premarket_data(df_2min)

    # Use 2-MIN for more granular pre-market levels
    # Captures every candle including 9:25-9:28AM
    premarket_high = pm_2min['high'].max() \
                     if len(pm_2min) > 0 else None
    premarket_low  = pm_2min['low'].min() \
                     if len(pm_2min) > 0 else None

    # ── Pre-market EMA touches (tracked not scored) ───────────
    pm_touch_5min  = False
    pm_touch_2min  = False
    pm_touch_time  = None
    pm_touch_price = None

    if len(pm_5min) > 0 and ema_5min:
        for idx, row in pm_5min.iterrows():
            if is_near_level(row['low'], ema_5min,
                             EMA_ALERT_PCT,
                             EMA_ALERT_MIN_DOLLAR):
                pm_touch_5min  = True
                pm_touch_time  = idx
                pm_touch_price = row['low']
                break

    if len(pm_2min) > 0 and ema_2min:
        for idx, row in pm_2min.iterrows():
            if is_near_level(row['low'], ema_2min,
                             EMA_ALERT_PCT,
                             EMA_ALERT_MIN_DOLLAR):
                pm_touch_2min = True
                if pm_touch_time is None:
                    pm_touch_time  = idx
                    pm_touch_price = row['low']
                break

    # ── Current price vs 200-EMAs ─────────────────────────────
    alert_5min = is_near_level(current_price, ema_5min,
                                EMA_ALERT_PCT,
                                EMA_ALERT_MIN_DOLLAR)
    alert_2min = is_near_level(current_price, ema_2min,
                                EMA_ALERT_PCT,
                                EMA_ALERT_MIN_DOLLAR)

    # ── VWAP confluence — 2-MIN only ──────────────────────────
    vwap_conf_5min = False
    vwap_conf_2min = (vwap is not None and
                      is_near_level(ema_2min, vwap,
                                    VWAP_CONFLUENCE_PCT,
                                    EMA_ALERT_MIN_DOLLAR))

    # ── Pre-market low confluence ─────────────────────────────
    pm_low_conf_5min = (premarket_low is not None and
                        is_near_level(ema_5min, premarket_low,
                                      PM_CONFLUENCE_PCT,
                                      EMA_ALERT_MIN_DOLLAR))
    pm_low_conf_2min = (premarket_low is not None and
                        is_near_level(ema_2min, premarket_low,
                                      PM_CONFLUENCE_PCT,
                                      EMA_ALERT_MIN_DOLLAR))

    # ── Pre-market high confluence ────────────────────────────
    pm_high_conf_5min = (premarket_high is not None and
                         is_near_level(ema_5min, premarket_high,
                                       PM_CONFLUENCE_PCT,
                                       EMA_ALERT_MIN_DOLLAR))
    pm_high_conf_2min = (premarket_high is not None and
                         is_near_level(ema_2min, premarket_high,
                                       PM_CONFLUENCE_PCT,
                                       EMA_ALERT_MIN_DOLLAR))

    # ── 200-EMA convergence ───────────────────────────────────
    emas_converging = is_near_level(ema_5min, ema_2min,
                                    EMA_CONVERGENCE_PCT,
                                    EMA_ALERT_MIN_DOLLAR)
    ema_conv_dist   = pct_distance(ema_5min, ema_2min)

    # ── 9-EMA triple confluence ───────────────────────────────
    ema9_confluence = (
        ema9_2min is not None and
        vwap is not None and
        is_near_level(ema9_2min, ema_2min,
                      EMA_CONVERGENCE_PCT,
                      EMA_ALERT_MIN_DOLLAR) and
        is_near_level(ema9_2min, vwap,
                      VWAP_CONFLUENCE_PCT,
                      EMA_ALERT_MIN_DOLLAR)
    )

    # ── Build alert data ──────────────────────────────────────
    alert_data = {
        # Timing
        'timestamp':               datetime.now().strftime(
                                   '%Y-%m-%d %H:%M:%S'),
        'ticker':                  ticker,
        'current_price':           round(current_price, 4),
        'open_price':              round(open_price, 4),

        # 5-MIN 200-EMA
        'ema200_5min':             round(ema_5min, 4),
        'dist_to_5min_ema_pct':    pct_distance(current_price,
                                                 ema_5min),
        '5min_ema_alert':          alert_5min,
        'vwap_confluence_5min':    vwap_conf_5min,
        'pm_low_confluence_5min':  pm_low_conf_5min,
        'pm_high_confluence_5min': pm_high_conf_5min,

        # 2-MIN 200-EMA
        'ema200_2min':             round(ema_2min, 4),
        'dist_to_2min_ema_pct':    pct_distance(current_price,
                                                 ema_2min),
        '2min_ema_alert':          alert_2min,
        'vwap_confluence_2min':    vwap_conf_2min,
        'pm_low_confluence_2min':  pm_low_conf_2min,
        'pm_high_confluence_2min': pm_high_conf_2min,

        # VWAP
        'vwap':                    round(vwap, 4) \
                                   if vwap else None,
        'dist_5min_to_vwap_pct':   pct_distance(ema_5min, vwap),
        'dist_2min_to_vwap_pct':   pct_distance(ema_2min, vwap),

        # Pre-market
        'premarket_high':          round(premarket_high, 4) \
                                   if premarket_high else None,
        'premarket_low':           round(premarket_low, 4) \
                                   if premarket_low else None,
        'pm_touch_5min':           pm_touch_5min,
        'pm_touch_2min':           pm_touch_2min,
        'pm_touch_time':           str(pm_touch_time) \
                                   if pm_touch_time else None,
        'pm_touch_price':          round(pm_touch_price, 4) \
                                   if pm_touch_price else None,

        # 200-EMA convergence
        'emas_converging':         emas_converging,
        'ema_convergence_dist_pct':ema_conv_dist,

        # 9-EMA triple confluence
        'ema9_2min':               round(ema9_2min, 4) \
                                   if ema9_2min else None,
        'ema9_confluence':         ema9_confluence,

        # Setup type — default standard
        # Blue Sky detected in place_momo_trade()
        # when we have actual open price at 9:30AM
        'setup_type':              'standard',

        # You fill in after market
        'paper_traded':            False,
        'outcome':                 '',
        'entry_price':             '',
        'exit_price':              '',
        'pnl':                     '',
        'notes':                   ''
    }

    # ── Score and type ────────────────────────────────────────
    alert_data['conviction_score'] = calculate_conviction_score(
        alert_data)
    alert_data['alert_type']       = determine_alert_type(
        alert_data)
    alert_data['attempt_count']    = attempts + 1

    # ── Only return if at least one EMA alert fires ───────────
    if not alert_5min and not alert_2min:
        return None

    # ── Update attempt counter — market hours only ────────────
    if current_time >= MARKET_OPEN:
        daily_attempts[ticker] = attempts + 1

    return alert_data


print("✅ Alert generator loaded")
print("   check_ticker(ticker) → alert_data or None")
print()
print("   Attempt rules:")
print("   Pre-market  → no attempts counted")
print("   9:30-9:40AM → max 2 attempts")
print()
print("   Blue Sky:")
print("   → NOT detected here")
print("   → Detected in place_momo_trade()")
print("   → When open price available at 9:30AM")

✅ Alert generator loaded
   check_ticker(ticker) → alert_data or None

   Attempt rules:
   Pre-market  → no attempts counted
   9:30-9:40AM → max 2 attempts

   Blue Sky:
   → NOT detected here
   → Detected in place_momo_trade()
   → When open price available at 9:30AM


In [24]:
# ─── Cell 10: Logger ──────────────────────────────────────────

def log_alert(alert_data):
    """
    Print alert to screen and save to CSV
    setup_type defaults to 'standard' here
    Updated to 'blue_sky' in place_momo_trade()
    if Blue Sky conditions confirmed at open
    """
    score  = alert_data['conviction_score']
    atype  = alert_data['alert_type']
    ticker = alert_data['ticker']
    price  = alert_data['current_price']

    # Conviction label
    if score >= 6:   conviction = "🔥 TRIPLE CONFLUENCE (6/6)"
    elif score >= 5: conviction = "💪 PERFECT (5/6)"
    elif score >= 4: conviction = "✅ VERY HIGH (4/6)"
    elif score >= 3: conviction = "⚠️  MODERATE (3/6)"
    elif score >= 2: conviction = "👀 LOW (2/6)"
    else:            conviction = "📝 WATCH (1/6)"

    print(f"\n{'='*60}")
    print(f"  🚨 200-EMA ALERT — {ticker}")
    print(f"  {alert_data['timestamp']}")
    print(f"{'='*60}")
    print(f"  Price:           ${price:.2f}")
    print(f"  Alert Type:      Type {atype}")
    print(f"  Conviction:      {conviction}")
    print(f"  Attempt:         #{alert_data['attempt_count']}/2")
    print(f"  Setup type:      {alert_data['setup_type']}"
          f" (Blue Sky confirmed at 9:30AM if applicable)")

    # EMA levels
    print(f"\n  EMA Levels:")
    print(f"  5-MIN 200-EMA:   ${alert_data['ema200_5min']:.4f}"
          f"  {'🎯' if alert_data['5min_ema_alert'] else '─'}"
          f"  ({alert_data['dist_to_5min_ema_pct']}% away)")
    print(f"  2-MIN 200-EMA:   ${alert_data['ema200_2min']:.4f}"
          f"  {'🎯' if alert_data['2min_ema_alert'] else '─'}"
          f"  ({alert_data['dist_to_2min_ema_pct']}% away)")
    if alert_data.get('ema9_2min'):
        print(f"  2-MIN 9-EMA:     ${alert_data['ema9_2min']:.4f}"
              f"  {'🔥' if alert_data.get('ema9_confluence') else '─'}")

    # Key levels
    print(f"\n  Key Levels:")
    if alert_data['vwap']:
        print(f"  VWAP:            ${alert_data['vwap']:.4f}")
    if alert_data['premarket_high']:
        print(f"  PM High:         "
              f"${alert_data['premarket_high']:.4f}"
              f"  ← target")
    if alert_data['premarket_low']:
        print(f"  PM Low:          "
              f"${alert_data['premarket_low']:.4f}")

    # Confluences
    print(f"\n  Confluences:")
    print(f"  5-MIN EMA alert:     "
          f"{'✅' if alert_data['5min_ema_alert'] else '─'}")
    print(f"  2-MIN EMA alert:     "
          f"{'✅' if alert_data['2min_ema_alert'] else '─'}")
    print(f"  200-EMAs converging: "
          f"{'✅' if alert_data['emas_converging'] else '─'}"
          f"  ({alert_data['ema_convergence_dist_pct']}% apart)")
    print(f"  VWAP near 2-MIN EMA: "
          f"{'✅' if alert_data['vwap_confluence_2min'] else '─'}")
    print(f"  PM low near 5-MIN:   "
          f"{'✅' if alert_data['pm_low_confluence_5min'] else '─'}")
    print(f"  PM low near 2-MIN:   "
          f"{'✅' if alert_data['pm_low_confluence_2min'] else '─'}")
    print(f"  9-EMA triple conf:   "
          f"{'🔥' if alert_data.get('ema9_confluence') else '─'}")
    print(f"  PM touch 5-MIN:      "
          f"{'✅' if alert_data['pm_touch_5min'] else '─'}"
          f"  {alert_data['pm_touch_time'] or ''}")
    print(f"  PM touch 2-MIN:      "
          f"{'✅' if alert_data['pm_touch_2min'] else '─'}")

    # Trade action
    print(f"\n  Action:")
    if score >= 4:
        print(f"  → PAPER TRADE will be placed (score {score}/6)")
        print(f"  → Blue Sky check runs at 9:30AM open")
    else:
        print(f"  → Alert only — logged to CSV (score {score}/6)")
    print(f"{'='*60}")

    # Save to CSV
    df_new = pd.DataFrame([alert_data])
    if os.path.exists(CSV_FILE):
        df_existing = pd.read_csv(CSV_FILE)
        df_combined = pd.concat([df_existing, df_new],
                                 ignore_index=True)
    else:
        df_combined = df_new

    df_combined.to_csv(CSV_FILE, index=False)
    print(f"  💾 Saved — {len(df_combined)} total alerts in CSV")


print("✅ Logger loaded")
print("   log_alert(alert_data) → prints + saves to CSV")

✅ Logger loaded
   log_alert(alert_data) → prints + saves to CSV


In [25]:
# ─── Cell 11: Paper Trade Executor ───────────────────────────
#
# TWO FRAMEWORKS:
#
# STANDARD (auto-traded):
#   Validated: $26,550 | $21.94/trade | 2024-2026
#   Size:    $100 risk
#   Entry:   Latest 2-MIN market close (NOT pre-market price)
#   Stop:    Attempt 1 → low of first market candle
#            Attempt 2 → LOD
#   Exit 1:  50% at 1R → stop moves to BE
#   Exit 2:  Trail 50% on prior 5-MIN low
#            Any price below prior 5-MIN low → exit
#   EOD:     4PM hard exit
#
# BLUE SKY (alert only — NOT auto-traded):
#   Not validated in backtest (104 trades, 63.5% stop rate)
#   Human judgment required:
#     → Tape confirmation
#     → Volume surge at EMA
#     → Clean bounce candle
#     → Strong catalyst
#   When detected:
#     → Prints alert to screen
#     → Logs to CSV with setup_type = 'blue_sky'
#     → NO auto order placed
#     → You trade discretionarily
#   Rules when trading manually:
#     Size:    $150 risk (1.5×)
#     Stop:    Opening low / 5-MIN EMA — NEVER moves
#     Exit 1:  50% at 1R
#     Exit 2:  Trail 50% on prior 5-MIN low
#     EOD:     4PM hard exit
#     Max:     2 attempts per stock per day
#
# KEY FIX — Aug 20 2026:
#   Entry price now uses ACTUAL market close
#   NOT the stale pre-market alert price
#   This fixes SKHX and AAPU type failures
#   where stop > entry due to stale price

momo_positions = {}


def detect_blue_sky(ticker, entry, ema_5min,
                    vwap, pm_high, df_2min_trade):
    """
    Detect Blue Sky condition at 9:30AM open

    Requirements:
      1. 5-MIN 200-EMA = opening low (within $0.05)
      2. VWAP at or below entry
      3. PM High at or below entry

    Returns:
      (is_blue_sky, opening_low, stop_price)
    """
    try:
        mkt_data = get_market_data(df_2min_trade)
        if len(mkt_data) == 0:
            return False, None, None

        opening_low    = mkt_data['low'].iloc[0]
        ema_equals_low = abs(ema_5min - opening_low) <= 0.05
        vwap_below     = vwap is None or vwap <= entry
        pmhigh_below   = (pm_high is None or
                          pd.isna(pm_high) or
                          pm_high <= entry)

        is_blue_sky = (ema_equals_low and
                       vwap_below and
                       pmhigh_below)

        stop = opening_low if is_blue_sky else None

        return is_blue_sky, opening_low, stop

    except Exception as e:
        print(f"   ⚠️  Blue Sky detection error: {e}")
        return False, None, None


def place_momo_trade(alert_data):
    """
    Main trade execution function

    Flow:
      1. Fetch 2-MIN market data
      2. Update entry to ACTUAL market close price
         (not stale pre-market alert price)
      3. Detect Blue Sky condition
      4. If Blue Sky → alert only, no order
      5. If Standard → validate and place paper trade

    Critical fix (Aug 20):
      alert_data['current_price'] = pre-market snapshot
      entry = mkt_data['close'].iloc[-1] = actual price
      This prevents stop > entry failures
    """
    ticker         = alert_data['ticker']
    premarket_high = alert_data['premarket_high']
    vwap           = alert_data['vwap']
    score          = alert_data['conviction_score']
    attempt        = alert_data['attempt_count']
    ema_5min       = alert_data['ema200_5min']

    # ── Score check ───────────────────────────────────────────
    if score < 4:
        print(f"   ⚠️  {ticker} — score {score}/6 "
              f"below threshold (need 4+)")
        return False

    # ── Already in position ───────────────────────────────────
    if ticker in momo_positions:
        print(f"   ⚠️  {ticker} — already in position")
        return False

    # ── No pre-market high ────────────────────────────────────
    if not premarket_high:
        print(f"   ⚠️  {ticker} — no pre-market high")
        return False

    try:
        # ── Fetch 2-MIN market data ───────────────────────────
        df_2min_trade = fetch_bars(ticker, 2)
        if df_2min_trade is None:
            print(f"   ⚠️  {ticker} — no 2-MIN data")
            return False

        mkt_data = get_market_data(df_2min_trade)
        if len(mkt_data) == 0:
            print(f"   ⚠️  {ticker} — no market data yet "
                  f"(waiting for 9:32AM first bar)")
            return False

        # ── CRITICAL FIX — Use actual market price ────────────
        # alert_data['current_price'] is a pre-market snapshot
        # By 9:30AM the price has moved significantly
        # Using stale price causes stop > entry failures
        # Always use the latest closed 2-MIN bar as entry
        entry = mkt_data['close'].iloc[-1]
        lod   = mkt_data['low'].min()

        # ── Detect Blue Sky at open ───────────────────────────
        is_blue_sky, opening_low, bs_stop = detect_blue_sky(
            ticker        = ticker,
            entry         = entry,
            ema_5min      = ema_5min,
            vwap          = vwap,
            pm_high       = premarket_high,
            df_2min_trade = df_2min_trade
        )

        # ── BLUE SKY — Alert only ─────────────────────────────
        if is_blue_sky:
            alert_data['setup_type']   = 'blue_sky'
            alert_data['paper_traded'] = False

            target_1r = entry + (entry - opening_low)

            print(f"\n{'='*60}")
            print(f"  🔥 BLUE SKY ALERT — {ticker}")
            print(f"  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")
            print(f"  Score:         {score}/6")
            print(f"  Entry area:    ${entry:.2f}")
            print(f"  5-MIN EMA:     ${ema_5min:.2f}")
            print(f"  Opening low:   ${opening_low:.2f}")
            print(f"  EMA gap:       "
                  f"${abs(ema_5min - opening_low):.4f} "
                  f"(max $0.05)")
            if vwap:
                print(f"  VWAP:          ${vwap:.2f} "
                      f"← below entry ✅")
            if premarket_high:
                print(f"  PM High:       "
                      f"${premarket_high:.2f} "
                      f"← below entry ✅")
            print(f"\n  ⚡ MANUAL TRADE RULES:")
            print(f"  Size:    $150 risk (1.5×)")
            print(f"  Entry:   ${entry:.2f} "
                  f"(second 2-MIN candle close)")
            print(f"  Stop:    ${opening_low:.2f} "
                  f"← NEVER moves")
            print(f"  1R:      ${target_1r:.2f}")
            print(f"  Exit 1:  50% at ${target_1r:.2f} (1R)")
            print(f"  Exit 2:  Trail 50% on prior 5-MIN low")
            print(f"  EOD:     4PM hard exit")
            print(f"\n  ⚠️  CHECK BEFORE ENTERING:")
            print(f"  → Tape showing buyers at "
                  f"${opening_low:.2f}?")
            print(f"  → Volume surge on bounce candle?")
            print(f"  → Clean green bounce candle?")
            print(f"  → Strong catalyst driving the stock?")
            print(f"\n  → NO AUTO ORDER PLACED")
            print(f"  → Trade discretionarily if confirmed")
            print(f"{'='*60}")
            return False

        # ── STANDARD — Validate stop ──────────────────────────
        # Stop based on attempt number
        if attempt == 1:
            # First attempt: low of first market candle
            stop = mkt_data['low'].iloc[0]
        else:
            # Second attempt: LOD
            stop = lod

        risk = entry - stop

        # Risk validation
        if risk <= 0:
            print(f"   ⚠️  {ticker} — stop ${stop:.2f} "
                  f"above entry ${entry:.2f} "
                  f"(stale price issue resolved — "
                  f"entry now ${entry:.2f})")
            return False

        if risk < entry * 0.002:
            print(f"   ⚠️  {ticker} — risk ${risk:.2f} "
                  f"too tight (min 0.2%)")
            return False

        if risk > entry * 0.05:
            print(f"   ⚠️  {ticker} — risk ${risk:.2f} "
                  f"too wide (max 5%)")
            return False

        # ── Position sizing ───────────────────────────────────
        shares = max(round(DOLLAR_RISK / risk), 1)

        # ── Build exit levels ─────────────────────────────────
        open_high = mkt_data['high'].max()

        raw_levels = {
            'vwap':      vwap,
            'open_high': open_high,
            'pm_high':   premarket_high
        }

        exit_levels = {
            name: price
            for name, price in raw_levels.items()
            if price is not None and
            not pd.isna(price) and
            price > entry
        }

        exit_levels = dict(
            sorted(exit_levels.items(),
                   key=lambda x: x[1]))

        if len(exit_levels) == 0:
            print(f"   ⚠️  {ticker} — no exit levels "
                  f"above entry ${entry:.2f}")
            print(f"          VWAP:      "
                  f"${vwap:.2f}" if vwap
                  else "          VWAP: N/A")
            print(f"          PM High:   "
                  f"${premarket_high:.2f}"
                  if premarket_high
                  else "          PM High: N/A")
            print(f"          Open High: ${open_high:.2f}")
            return False

        # ── Assign exit percentages ───────────────────────────
        level_names  = list(exit_levels.keys())
        level_prices = list(exit_levels.values())

        if len(level_names) == 1:
            exit_plan = [
                (level_prices[0], level_names[0], 1.0)
            ]
        elif len(level_names) == 2:
            exit_plan = [
                (level_prices[0], level_names[0], 0.50),
                (level_prices[1], level_names[1], 1.00),
            ]
        else:
            exit_plan = [
                (level_prices[0], level_names[0], 0.50),
                (level_prices[1], level_names[1], 0.50),
                (level_prices[2], level_names[2], 1.00),
            ]

        # ── Place buy order ───────────────────────────────────
        order = trading_client.submit_order(
            MarketOrderRequest(
                symbol        = ticker,
                qty           = shares,
                side          = OrderSide.BUY,
                time_in_force = TimeInForce.DAY
            )
        )

        # ── Store position ────────────────────────────────────
        momo_positions[ticker] = {
            'order_id':       order.id,
            'entry':          entry,
            'stop':           stop,
            'risk':           risk,
            'shares_total':   shares,
            'shares_remain':  shares,
            'framework':      'standard',
            'attempt':        attempt,
            'entry_time':     datetime.now(),
            'conviction':     score,
            'exit_plan':      exit_plan,
            'exit_index':     0,
            'stop_at_be':     False,
            'tp1_hit':        False,
            'prior_5min_low': None,
        }

        alert_data['setup_type']   = 'standard'
        alert_data['paper_traded'] = True

        # ── Print confirmation ────────────────────────────────
        print(f"\n🟢 STANDARD PAPER TRADE — {ticker}")
        print(f"   Score:         {score}/6")
        print(f"   Attempt:       #{attempt}/2")
        print(f"   Shares:        {shares}")
        print(f"   Entry:         ${entry:.2f} "
              f"← actual market close")
        print(f"   Alert price:   "
              f"${alert_data['current_price']:.2f} "
              f"← pre-market snapshot (not used)")
        if attempt == 1:
            print(f"   Stop:          ${stop:.2f} "
                  f"← first candle low")
        else:
            print(f"   Stop:          ${stop:.2f} "
                  f"← LOD (second attempt)")
        print(f"   Risk:          ${risk:.2f} × "
              f"{shares} = ${risk * shares:.2f}")
        print(f"   1R target:     ${entry + risk:.2f}")
        print(f"   After 1R:      stop → BE ${entry:.2f}")
        print(f"   After 1R:      trail {shares // 2} "
              f"shares on prior 5-MIN low")
        print(f"\n   Exit Plan (nearest → furthest):")
        remaining = shares
        for i, (price, name, pct) in enumerate(exit_plan):
            if i < len(exit_plan) - 1:
                sell = round(remaining * pct)
            else:
                sell = remaining
            label = name.replace('_', ' ').upper()
            print(f"   L{i+1} {label:<12} "
                  f"${price:.2f} → sell {sell} shares "
                  f"(+${(price - entry) * sell:.2f})")
            remaining -= sell
        print(f"   Order ID:      {order.id}")
        return True

    except Exception as e:
        error = str(e)
        if 'buying_power' in error or \
                'insufficient' in error.lower():
            print(f"   💰 {ticker} — buying power "
                  f"exhausted — skipping")
        else:
            print(f"   ❌ Order failed {ticker}: {e}")
        return False


def manage_momo_positions():
    """
    Monitor all open STANDARD positions every scan

    Standard framework:
      → Check 1R target (50% exit)
      → After TP1: stop moves to BE
      → Trail remaining 50% on prior 5-MIN low
      → Any price below prior 5-MIN low → exit
      → Stop hit → exit all
      → EOD 4PM → exit all

    Note: Blue Sky positions are NOT managed here
          They are traded and managed manually
    """
    if not momo_positions:
        return

    for ticker, pos in list(momo_positions.items()):
        try:
            # ── Fetch latest data ─────────────────────────────
            df_2min = fetch_bars(ticker, 2)
            df_5min = fetch_bars(ticker, 5)

            if df_2min is None or df_5min is None:
                continue

            mkt_2min = get_market_data(df_2min)
            mkt_5min = get_market_data(df_5min)

            if len(mkt_2min) == 0:
                continue

            price         = mkt_2min['close'].iloc[-1]
            shares_remain = pos['shares_remain']
            entry         = pos['entry']
            risk          = pos['risk']
            target_1r     = entry + risk
            tp1_hit       = pos['tp1_hit']

            if shares_remain <= 0:
                if ticker in momo_positions:
                    del momo_positions[ticker]
                continue

            # ── Update prior 5-MIN low ────────────────────────
            if len(mkt_5min) >= 2:
                pos['prior_5min_low'] = \
                    mkt_5min['low'].iloc[-2]

            prior_5min_low = pos['prior_5min_low']

            # ── Current active stop ───────────────────────────
            current_stop = entry if pos['stop_at_be'] \
                else pos['stop']

            # ── EOD check ─────────────────────────────────────
            if datetime.now().time() >= EOD:
                trading_client.submit_order(
                    MarketOrderRequest(
                        symbol        = ticker,
                        qty           = shares_remain,
                        side          = OrderSide.SELL,
                        time_in_force = TimeInForce.DAY
                    )
                )
                pnl = (price - entry) * shares_remain
                print(f"\n🔔 EOD EXIT — {ticker} "
                      f"@ ${price:.2f} | "
                      f"P&L: ${pnl:.2f}")
                del momo_positions[ticker]
                continue

            # ── Check stop ────────────────────────────────────
            if price <= current_stop:
                trading_client.submit_order(
                    MarketOrderRequest(
                        symbol        = ticker,
                        qty           = shares_remain,
                        side          = OrderSide.SELL,
                        time_in_force = TimeInForce.DAY
                    )
                )
                pnl = (current_stop - entry) * shares_remain
                stop_type = "BE" if pos['stop_at_be'] \
                    else "original"
                print(f"\n🔴 STOP HIT ({stop_type}) — "
                      f"{ticker} @ ${current_stop:.2f} | "
                      f"P&L: ${pnl:.2f}")
                del momo_positions[ticker]
                continue

            # ── Check 1R target (50% exit) ────────────────────
            if not tp1_hit and price >= target_1r:
                lot_a = round(shares_remain * 0.50)
                lot_a = min(lot_a, shares_remain)

                if lot_a > 0:
                    trading_client.submit_order(
                        MarketOrderRequest(
                            symbol        = ticker,
                            qty           = lot_a,
                            side          = OrderSide.SELL,
                            time_in_force = TimeInForce.DAY
                        )
                    )
                    pnl = (target_1r - entry) * lot_a
                    print(f"\n✅ TP1 — {ticker} "
                          f"@ ${target_1r:.2f} | "
                          f"sold {lot_a} shares | "
                          f"P&L: +${pnl:.2f}")
                    pos['shares_remain'] -= lot_a
                    pos['tp1_hit']        = True
                    pos['stop_at_be']     = True
                    print(f"   Stop → BE: ${entry:.2f}")
                    print(f"   Trailing "
                          f"{pos['shares_remain']} "
                          f"shares on prior 5-MIN low")

            # ── Trail on prior 5-MIN low ──────────────────────
            elif tp1_hit and shares_remain > 0:
                if (prior_5min_low is not None and
                        price < prior_5min_low):
                    trading_client.submit_order(
                        MarketOrderRequest(
                            symbol        = ticker,
                            qty           = shares_remain,
                            side          = OrderSide.SELL,
                            time_in_force = TimeInForce.DAY
                        )
                    )
                    pnl = (price - entry) * shares_remain
                    print(f"\n✅ TRAIL EXIT — {ticker} "
                          f"@ ${price:.2f} | "
                          f"P&L: ${pnl:.2f}")
                    del momo_positions[ticker]
                    continue

            # ── Status update ─────────────────────────────────
            if shares_remain > 0 and \
                    ticker in momo_positions:
                unrealized = (price - entry) * shares_remain
                stop_label = "BE" if pos['stop_at_be'] \
                    else "orig"
                trail_label = f"${prior_5min_low:.2f}" \
                    if prior_5min_low else "building"
                print(f"   🟢 {ticker} ${price:.2f} | "
                      f"{shares_remain} shares | "
                      f"unrealized: ${unrealized:.2f} | "
                      f"stop ({stop_label}): "
                      f"${current_stop:.2f} | "
                      f"trail: {trail_label}")

        except Exception as e:
            print(f"   ❌ Position error {ticker}: {e}")


print("✅ Paper trade executor loaded")
print()
print("   STANDARD (auto-traded):")
print("   → Score 4+ required")
print("   → $100 risk per trade")
print("   → Entry = actual 2-MIN market close")
print("     (NOT stale pre-market alert price)")
print("   → Attempt 1 stop: first candle low")
print("   → Attempt 2 stop: LOD")
print("   → 50% at 1R → stop to BE")
print("   → Trail 50% on prior 5-MIN low")
print("   → Any price below 5-MIN low → exit")
print()
print("   BLUE SKY (alert only):")
print("   → Detected at 9:30AM open")
print("   → Condition: 5-MIN EMA = opening low (±$0.05)")
print("   →            ALL resistance at/below entry")
print("   → NO auto order — trade discretionarily")
print("   → Check tape + volume before entering")
print("   → Manual rules: $150 risk, fixed stop,")
print("                   50% at 1R, trail 5-MIN")
print()
print("   KEY FIX (Aug 20 2026):")
print("   → Entry now uses actual market close")
print("   → Prevents stop > entry failures")
print("   → Fixes SKHX and AAPU type misses")

✅ Paper trade executor loaded

   STANDARD (auto-traded):
   → Score 4+ required
   → $100 risk per trade
   → Entry = actual 2-MIN market close
     (NOT stale pre-market alert price)
   → Attempt 1 stop: first candle low
   → Attempt 2 stop: LOD
   → 50% at 1R → stop to BE
   → Trail 50% on prior 5-MIN low
   → Any price below 5-MIN low → exit

   BLUE SKY (alert only):
   → Detected at 9:30AM open
   → Condition: 5-MIN EMA = opening low (±$0.05)
   →            ALL resistance at/below entry
   → NO auto order — trade discretionarily
   → Check tape + volume before entering
   → Manual rules: $150 risk, fixed stop,
                   50% at 1R, trail 5-MIN

   KEY FIX (Aug 20 2026):
   → Entry now uses actual market close
   → Prevents stop > entry failures
   → Fixes SKHX and AAPU type misses


In [26]:
# ─── Cell 12: CSV Viewer ──────────────────────────────────────

def view_alerts(min_conviction=0, ticker=None,
                today_only=False, setup_type=None):
    """
    Review logged alerts
    min_conviction: filter by minimum score
    ticker:         filter by specific stock
    today_only:     only show today's alerts
    setup_type:     'standard', 'blue_sky', or None for all
    """
    if not os.path.exists(CSV_FILE):
        print("❌ No alerts logged yet")
        return

    df = pd.read_csv(CSV_FILE)
    if len(df) == 0:
        print("❌ No alerts logged yet")
        return

    # Apply filters
    if min_conviction > 0:
        df = df[df['conviction_score'] >= min_conviction]
    if ticker:
        df = df[df['ticker'] == ticker.upper()]
    if today_only:
        today = datetime.now().strftime('%Y-%m-%d')
        df    = df[df['timestamp'].str.startswith(today)]
    if setup_type:
        df = df[df['setup_type'] == setup_type]

    print(f"\n{'='*70}")
    print(f"  200-EMA MOMO ALERT DATABASE")
    print(f"  Total alerts shown: {len(df)}")
    print(f"{'='*70}")

    if len(df) == 0:
        print("  No alerts match filters")
        return

    # Conviction breakdown
    print(f"\n  Conviction breakdown:")
    for score in range(6, 0, -1):
        count = len(df[df['conviction_score'] == score])
        if count > 0:
            bar = '█' * count
            print(f"  Score {score}: {bar} ({count})")

    # Setup type breakdown
    print(f"\n  Setup type breakdown:")
    if 'setup_type' in df.columns:
        std_count = len(df[df['setup_type'] == 'standard'])
        bs_count  = len(df[df['setup_type'] == 'blue_sky'])
        print(f"  Standard:   {std_count}")
        print(f"  Blue Sky:   {bs_count}")

    # Alert type breakdown
    print(f"\n  Alert types:")
    type_names = {
        1: 'Type 1 — 5-MIN only',
        2: 'Type 2 — 2-MIN only',
        3: 'Type 3 — Both EMAs',
        4: 'Type 4 — EMA + VWAP',
        5: 'Type 5 — EMA + PM low',
        6: 'Type 6 — Triple confluence'
    }
    for atype, name in type_names.items():
        count = len(df[df['alert_type'] == atype])
        if count > 0:
            print(f"  {name}: {count}")

    # Paper traded
    if 'paper_traded' in df.columns:
        traded = df['paper_traded'].sum()
        print(f"\n  Paper traded: {traded} setups")

    # Recent alerts
    print(f"\n  Recent alerts (last 10):")
    cols = ['timestamp', 'ticker', 'alert_type',
            'conviction_score', 'setup_type',
            'current_price', 'ema200_5min', 'ema200_2min',
            'emas_converging', 'vwap_confluence_2min',
            'paper_traded', 'outcome', 'pnl']
    available = [c for c in cols if c in df.columns]
    print(df[available].tail(10).to_string(index=False))
    print(f"\n  📁 CSV: {CSV_FILE}")
    print(f"  💡 Open in Excel to add outcome/pnl/notes")
    print(f"\n  Filter options:")
    print(f"  view_alerts(min_conviction=4)")
    print(f"  view_alerts(today_only=True)")
    print(f"  view_alerts(setup_type='blue_sky')")
    print(f"  view_alerts(ticker='SNAP')")


view_alerts()


  200-EMA MOMO ALERT DATABASE
  Total alerts shown: 567

  Conviction breakdown:
  Score 6: ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ (184)
  Score 5: █████████████████████████████████████████████████████████████████████████████████████████████████████████████████ (113)
  Score 4: ███████████████████████████████████████████████████████████████████████████████████████████████████ (99)
  Score 3: ████████████████████████████████████████████████████████████████████████████████████████████████████████ (104)
  Score 2: ██████████████████████████████████████████ (42)
  Score 1: █████████████████████████ (25)

  Setup type breakdown:
  Standard:   567
  Blue Sky:   0

  Alert types:
  Type 1 — 5-MIN only: 14
  Type 2 — 2-MIN only: 11
  Type 3 — Both EMAs: 17
  Type 4 — EMA + VWAP: 23
  Type 5 — EMA + PM low: 94
  Type 6 — Triple confluence: 408

  Pap

In [27]:
# ─── Cell 13: Main Scanner Loop ───────────────────────────────

def run_alert_tracker(test_mode=False):
    """
    Main monitoring loop

    Session flow:
      5:00AM - 9:30AM → Pre-market monitoring
                         Detect 200-EMA touches
                         Log to CSV for research
                         NO attempt counting
                         NO paper trades
                         Scans every 5 minutes

      9:30AM - 9:40AM → Alert window
                         Paper trades fire here
                         Scans every 2 MINUTES
                         Attempts ONLY count when
                         order actually placed in Alpaca
                         Score 4+ required to trade
                         All tickers start fresh at 0 attempts

      9:40AM - 4:00PM → Position management
                         Monitor open positions
                         Execute exits at levels
                         Stop at LOD
                         EOD close at 4PM
                         Scans every 5 minutes

    test_mode=True  → 3 scans then stop
    test_mode=False → runs until 4PM EOD

    KEY FIX (Aug 20 2026):
      Attempt counter now ONLY increments when
      an order is actually placed in Alpaca
      Failed place_momo_trade() calls do NOT
      consume an attempt
      This prevents lockout from market data
      timing issues at 9:30AM open
    """
    print(f"{'='*60}")
    print(f"  🔍 200-EMA MOMO ALERT TRACKER")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}\n")

    reset_daily_attempts()

    iteration = 0
    last_day  = datetime.now().date()

    while True:
        iteration    += 1
        now           = datetime.now()
        current_time  = now.time()

        # ── Reset at new trading day ──────────────────────────
        if now.date() != last_day:
            reset_daily_attempts()
            last_day = now.date()
            print(f"\n🌅 New trading day — counters reset")

        # ── Load watchlist fresh each scan ────────────────────
        watchlist = load_watchlist()
        if not watchlist:
            print("⚠️  Watchlist empty — add tickers to "
                  "200_watchlist.txt")
            time_module.sleep(60)
            continue

        print(f"\n{'─'*60}")
        print(f"  Scan #{iteration} — "
              f"{now.strftime('%H:%M:%S')}")
        print(f"  Watchlist: {watchlist}")
        print(f"{'─'*60}")

        # ── Session flags ─────────────────────────────────────
        in_premarket    = PREMARKET_START <= current_time \
                          < MARKET_OPEN
        in_alert_window = MARKET_OPEN <= current_time \
                          <= ALERT_WINDOW_END
        past_window     = current_time > ALERT_WINDOW_END
        past_eod        = current_time >= EOD

        # ── EOD — close everything and stop ──────────────────
        if past_eod:
            if momo_positions:
                print("🔔 EOD — closing all positions")
                manage_momo_positions()
            print("🔔 Market closed — stopping tracker")
            break

        # ── Past alert window — manage positions only ─────────
        if past_window:
            print(f"⏰ Alert window closed "
                  f"({ALERT_WINDOW_END})")
            if momo_positions:
                print(f"   Managing "
                      f"{len(momo_positions)} "
                      f"open position(s)...")
                manage_momo_positions()
            else:
                print(f"   No open positions")
                print(f"   Run Cell 12 to review "
                      f"today's alerts")
                break

        # ── Pre-market (5AM - 9:30AM) ─────────────────────────
        # Purpose: data collection only
        # Rules:
        #   → Detect 200-EMA touches
        #   → Log all alerts to CSV
        #   → NO attempt counting
        #   → NO paper trades
        #   → All tickers scan every time
        #   → All tickers start fresh at 9:30AM
        if in_premarket:
            print(f"🌙 Pre-market "
                  f"({PREMARKET_START}-{MARKET_OPEN})")
            print(f"   Collecting data — no trades yet")
            print(f"   All tickers start fresh "
                  f"at 9:30AM\n")

            for ticker in watchlist:
                alert = check_ticker(ticker)

                if alert and (alert['pm_touch_5min'] or
                              alert['pm_touch_2min']):
                    score = alert['conviction_score']
                    print(f"\n🌙 PRE-MARKET TOUCH: "
                          f"{ticker} (score {score}/6)")
                    log_alert(alert)

        # ── Alert window (9:30AM - 9:40AM) ───────────────────
        # Purpose: paper trading window
        # Rules:
        #   → All tickers start with 0 attempts
        #   → Score 4+ → paper trade attempted
        #   → Attempt ONLY counts if order placed ← KEY FIX
        #   → Score 1-3 → alert logged only
        #   → Max 2 SUCCESSFUL attempts per ticker
        #   → Scans every 2 minutes
        elif in_alert_window:
            print(f"🔔 ALERT WINDOW ACTIVE (9:30-9:40AM)")
            print(f"   Scanning every 2 minutes")
            print(f"   Paper trades firing for score 4+")
            print(f"   Attempts only count on successful "
                  f"orders\n")
            alerts_fired = 0

            for ticker in watchlist:
                # Check attempt limit
                # Only counts SUCCESSFUL orders
                # Failed place_momo_trade() = attempt NOT used
                attempts = daily_attempts.get(ticker, 0)
                if attempts >= 2:
                    print(f"   {ticker:<8} → max attempts "
                          f"({attempts}/2 orders placed)")
                    continue

                print(f"   Checking {ticker}...")
                alert = check_ticker(ticker)

                if alert:
                    alerts_fired += 1
                    log_alert(alert)

                    if alert['conviction_score'] >= 4:
                        # Wait for market data to be available
                        time_module.sleep(2)
                        traded = place_momo_trade(alert)

                        if traded:
                            # ── ONLY increment on success ──
                            # This is the key fix:
                            # Failed orders don't use attempts
                            # Market data timing failures
                            # don't lock out the ticker
                            daily_attempts[ticker] = \
                                attempts + 1
                            alert['paper_traded'] = True
                            current_attempts = \
                                daily_attempts[ticker]
                            print(f"   🟢 Order placed — "
                                  f"{ticker} "
                                  f"(attempt "
                                  f"{current_attempts}/2)")
                        else:
                            print(f"   ⚠️  {ticker} — "
                                  f"order failed, attempt "
                                  f"NOT counted "
                                  f"({attempts}/2 used)")
                    else:
                        print(f"   📝 Score "
                              f"{alert['conviction_score']}"
                              f"/6 — logged only (need 4+)")
                else:
                    print(f"   {ticker:<8} → no alert")

            if alerts_fired == 0:
                print(f"\n  ✅ No setups this scan")

        # ── Too early ─────────────────────────────────────────
        else:
            print(f"⏳ Waiting for pre-market "
                  f"({PREMARKET_START})")

        # ── Manage open positions every scan ──────────────────
        if momo_positions:
            print(f"\n  📋 {len(momo_positions)} "
                  f"open position(s):")
            manage_momo_positions()

        # ── Test mode — stop after 3 scans ────────────────────
        if test_mode and iteration >= 3:
            print(f"\n✅ Test complete — {iteration} scans")
            break

        # ── Sleep between scans ───────────────────────────────
        # 2 minutes during alert window
        #   → Faster scanning catches entries
        #   → 5 scans in 10-minute window
        # 5 minutes all other times
        if in_alert_window:
            print(f"\n  💤 Next scan in 2 minutes "
                  f"(alert window)...")
            time_module.sleep(120)
        else:
            print(f"\n  💤 Next scan in 5 minutes...")
            time_module.sleep(300)

    print(f"\n{'='*60}")
    print(f"  🏁 TRACKER STOPPED")
    print(f"  Run Cell 12 → view_alerts() to review")
    print(f"{'='*60}")


print("✅ Scanner loaded")
print("   run_alert_tracker()               → live mode")
print("   run_alert_tracker(test_mode=True) → 3 scans only")
print()
print("   Session schedule:")
print("   5:00AM - 9:30AM → Pre-market (5-min scans)")
print("   9:30AM - 9:40AM → Paper trades (2-min scans)")
print("   9:40AM - 4:00PM → Position mgmt (5-min scans)")
print()
print("   Attempt counter rules (KEY FIX Aug 20):")
print("   → Attempts ONLY count on successful orders")
print("   → Failed place_momo_trade() = attempt NOT used")
print("   → Market data timing failures don't lock ticker")
print("   → Max 2 PLACED orders per ticker per day")

✅ Scanner loaded
   run_alert_tracker()               → live mode
   run_alert_tracker(test_mode=True) → 3 scans only

   Session schedule:
   5:00AM - 9:30AM → Pre-market (5-min scans)
   9:30AM - 9:40AM → Paper trades (2-min scans)
   9:40AM - 4:00PM → Position mgmt (5-min scans)

   Attempt counter rules (KEY FIX Aug 20):
   → Attempts ONLY count on successful orders
   → Failed place_momo_trade() = attempt NOT used
   → Market data timing failures don't lock ticker
   → Max 2 PLACED orders per ticker per day


In [28]:
# ─── Cell 14: START ───────────────────────────────────────────
# Run ALL cells first (Ctrl+F9)
# Then run this cell to start live tracking
#
# Live mode — runs until 4PM:
run_alert_tracker()
#
# Test mode — 3 scans only (for testing):
# run_alert_tracker(test_mode=True)

  🔍 200-EMA MOMO ALERT TRACKER
  2026-08-31 13:53:17

✅ Daily attempt counters reset

────────────────────────────────────────────────────────────
  Scan #1 — 13:53:17
  Watchlist: ['SNAP', 'MRVL', 'DKNG', 'PLTU', 'SMCX', 'NVDL', 'SNXX', 'AMZU', 'MSFU', 'AAPU', 'TSLL', 'SPCH', 'SSPC', 'MRNA', 'SKHX', 'MVLL', 'IBIT', 'CONL', 'AMDL', 'HOOG', 'SNDQ', 'CMG', 'CRMG', 'AFRM', 'GME']
────────────────────────────────────────────────────────────
⏰ Alert window closed (09:40:00)
   No open positions
   Run Cell 12 to review today's alerts

  🏁 TRACKER STOPPED
  Run Cell 12 → view_alerts() to review
